In [123]:
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt 

In [124]:
df=pd.read_csv("housing.csv")

In [125]:
df['income_cat']=pd.cut(df['median_income'],bins=[0, 1.5, 3.0, 4.5, 6.0 , np.inf] , labels=[1,2,3,4,5])

In [126]:
from sklearn.model_selection import StratifiedShuffleSplit

split= StratifiedShuffleSplit(n_splits=1 , test_size=0.2 , random_state=42)


for train_index , test_index in split.split(df ,df['income_cat']):

    strat_train_set=df.loc[train_index]  ## its a original dataframe ie source

    strat_test_set=df.loc[test_index]     ##its a original dataframe ie source 

In [127]:
#let remove income_cat col

for sett in (strat_train_set , strat_test_set):
    
    sett.drop('income_cat' , axis=1, inplace=True)

In [128]:
df=strat_train_set.copy()

In [129]:
 df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16512 entries, 12655 to 19773
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           16512 non-null  float64
 1   latitude            16512 non-null  float64
 2   housing_median_age  16512 non-null  float64
 3   total_rooms         16512 non-null  float64
 4   total_bedrooms      16354 non-null  float64
 5   population          16512 non-null  float64
 6   households          16512 non-null  float64
 7   median_income       16512 non-null  float64
 8   median_house_value  16512 non-null  float64
 9   ocean_proximity     16512 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.4+ MB


In [130]:
housing=strat_train_set.drop('median_house_value' ,axis=1 ) ##features (did not write inplace=true not want change in original dataset ie strat_train)

housing_labels= strat_train_set['median_house_value'].copy()   ##labels

In [131]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")              ## imputer is now a robot will do everyting, here strategy is median , can be mean ....

## housing_num= housing.select_dtypes(include=[np.number])  ## want to add categorical data to give fix no.  ie - ocean proximity , so i add  comment 
                                                              

imputer.fit(housing_num)                                 ##now robot is analyze all table value 

SimpleImputer(strategy='median')

In [132]:
x=imputer.transform(housing_num)                         ## now robot fill nan with mdeian nd made new table x  and  convert into np array 

In [133]:
x


array([[-1.2146e+02,  3.8520e+01,  2.9000e+01, ...,  2.2370e+03,
         7.0600e+02,  2.1736e+00],
       [-1.1723e+02,  3.3090e+01,  7.0000e+00, ...,  2.0150e+03,
         7.6800e+02,  6.3373e+00],
       [-1.1904e+02,  3.5370e+01,  4.4000e+01, ...,  6.6700e+02,
         3.0000e+02,  2.8750e+00],
       ...,
       [-1.2272e+02,  3.8440e+01,  4.8000e+01, ...,  4.5800e+02,
         1.7200e+02,  3.1797e+00],
       [-1.2270e+02,  3.8310e+01,  1.4000e+01, ...,  1.2080e+03,
         5.0100e+02,  4.1964e+00],
       [-1.2214e+02,  3.9970e+01,  2.7000e+01, ...,  6.2500e+02,
         1.9700e+02,  3.1319e+00]])

In [134]:
housing=pd.DataFrame(x ,columns=housing_num.columns , index=housing_num.index)  # convert np arr ie x again to housing  dataframe

In [135]:
housing['ocean_proximity']= df['ocean_proximity']           #add caterogical data again in housing dataframe , as NAN is filled now with median 


In [136]:
housing = housing[['ocean_proximity']]  #this is done for one  hot encoder we only include category col as it divide into subpart - inland , bay ...


In [137]:
housing      

,ocean_proximity
12655,INLAND
15502,NEAR OCEAN
2908,INLAND
14053,NEAR OCEAN
20496,<1H OCEAN
...,...
15174,<1H OCEAN
12661,INLAND
19263,<1H OCEAN
19140,<1H OCEAN


In [138]:
set(housing['ocean_proximity'])

{'<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'}

In [139]:
# ORDINAL ENCODING IS USED WHEN THERE IS ORDER MEANS EACH DATA IS RALTED WE GO FAR POWER REDUCES LIKE - ONLINE RATING so model will learn 

#   that pattrern like replacing value in inc order - 1H OCEAN -1 ,  INLAND-2 ,  ISLAND-3 , NEAR BAY-4 , NEAR OCEAN-5


In [140]:
# from sklearn.preprocessing import OrdinalEncoder

In [141]:
# ordinal_encoder= OrdinalEncoder()

# housing_cat=  ordinal_encoder.fit_transform(housing)   ## convert into numpy array 

In [142]:
# housing_cat = pd.DataFrame(housing_cat , columns= housing.columns ,index=housing.index)

In [143]:
# ONE HOT ENCODER FOR UNORDER 

In [144]:
from sklearn.preprocessing import OneHotEncoder

In [145]:
hot_encoder=OneHotEncoder()

In [146]:
housing_cat=hot_encoder.fit_transform(housing)

In [147]:
housing_cat.toarray()

array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0.],
       ...,
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.]])

In [148]:
hot_encoder.categories_

[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
       dtype=object)]

In [149]:
housing_cat=pd.DataFrame(housing_cat.toarray() , columns=['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'] ,index=housing.index)

In [150]:
housing_cat

,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
12655,0.0,1.0,0.0,0.0,0.0
15502,0.0,0.0,0.0,0.0,1.0
2908,0.0,1.0,0.0,0.0,0.0
14053,0.0,0.0,0.0,0.0,1.0
20496,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...
15174,1.0,0.0,0.0,0.0,0.0
12661,0.0,1.0,0.0,0.0,0.0
19263,1.0,0.0,0.0,0.0,0.0
19140,1.0,0.0,0.0,0.0,0.0
